# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore the FAIR^2 dataset, which contains tabular clinicopathological and molecular data for 77 cancer survivors with second primary colorectal cancer. All dataset entities (record sets, fields, columns, etc.) are referenced by their `@id` for clarity and reproducibility.

### Dataset Source
The dataset is defined and described by its Croissant schema, available at:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure mlcroissant is installed
!pip install -q mlcroissant

## 1. Data Loading

Load the dataset and metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

List available record sets and their fields (with their `@id`). This is useful for selecting which structures to extract next.

In [ ]:
# Explore record sets using the Croissant metadata API
from pprint import pprint

# Retrieve all available record sets by @id
record_sets = list(dataset.record_sets.keys())

print('Available record sets by @id:')
for rs_id in record_sets:
    rs = dataset.record_sets[rs_id]
    print(f"- {rs_id} (name: {getattr(rs, 'name', None)})")

    # Show fields for each record set
    fields = rs.fields if hasattr(rs, 'fields') else []
    print("  Fields:")
    for f_id in fields:
        field = dataset.fields[f_id] if f_id in dataset.fields else None
        if field:
            print(f"    - {f_id} (name: {getattr(field, 'name', None)}, dataType: {getattr(field, 'dataType', None)})")
        else:
            print(f"    - {f_id} (field metadata not found)")

## 3. Data Extraction

Extract tabular records from the primary record set as a pandas DataFrame for further analysis. All `@id` values are taken from the overview above.

In [ ]:
# For this dataset, assume the primary tabular record set @id is as follows:
# (You may need to update this to match the correct @id from the printed overview above.)
tabular_record_set_id = None
for rs_id in record_sets:
    rs = dataset.record_sets[rs_id]
    # Heuristic: pick the first record set that contains fields (tabular data)
    if hasattr(rs, 'fields') and len(rs.fields) > 0:
        tabular_record_set_id = rs_id
        break

if tabular_record_set_id is None:
    raise RuntimeError("No record set with fields found.")

print(f"Using record set @id: {tabular_record_set_id}\n---")

# Pull all records
records = list(dataset.records(record_set=tabular_record_set_id))
df = pd.DataFrame(records)

print(f"Columns in `{tabular_record_set_id}`:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)

Let's process and analyze the data: Filter based on a numeric field, normalize values, and optionally group by a categorical field.

All field and column references use their Croissant `@id`s.


In [ ]:
# Infer a typical numeric field @id (e.g., 'age', 'interval', etc.) from field names, else pick an integer/float field.
import numpy as np

# Identify likely numeric fields
numeric_candidates = []
for col in df.columns:
    if np.issubdtype(df[col].dropna().apply(type).mode()[0], np.number):
        numeric_candidates.append(col)
    elif ('age' in col.lower()) or ('interval' in col.lower()):
        numeric_candidates.append(col)
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    numeric_field_id = df.select_dtypes(include=np.number).columns[0] if len(df.select_dtypes(include=np.number).columns) else df.columns[0]

print(f"Selected numeric field for EDA: {numeric_field_id}")

# Choose a group-by field (categorical, e.g. 'sex', 'msi_status', etc.)
group_field_id = None
for col in df.columns:
    if df[col].dtype == 'object' and (('sex' in col.lower()) or ('msi' in col.lower()) or ('anatomical' in col.lower()) or ('location' in col.lower())):
        group_field_id = col
        break
if group_field_id is None:
    # fallback to the first object dtype column
    cand = df.select_dtypes(include=object).columns
    if len(cand):
        group_field_id = cand[0]
print(f"Group field for EDA: {group_field_id}")

# Remove clearly erroneous or null rows
filtered_df = df.copy()
filtered_df = filtered_df[filtered_df[numeric_field_id].notnull()]

# Define threshold as mean for demo
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    threshold = filtered_df[numeric_field_id].mean() if not np.isnan(filtered_df[numeric_field_id].mean()) else 1
else:
    threshold = 1  # Dummy value

filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
print(f"Filtered records in `{tabular_record_set_id}` where {numeric_field_id} > {threshold:.1f}:")
print(filtered_df[[numeric_field_id]].head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())/filtered_df[numeric_field_id].std()
)
print(f"\nNormalized {numeric_field_id} values (first few rows):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by the chosen field if present
if group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped)
else:
    print("No suitable group field for grouping found.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and, if available, compare it across groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.tight_layout()
plt.show()

# Boxplot by group if available
if group_field_id in df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR^2 dataset on clinical and molecular characteristics of second primary colorectal cancer survivors using `mlcroissant`. We identified the record sets, loaded records by their Croissant `@id`, and performed basic exploratory analysis. Numeric values were normalized and filtered, and group-wise summaries were presented.

**Note:** For detailed clinical interpretation, consult the full data dictionary and metadata for proper use of field semantics.
